# 09 — El mismo agente, en LangChain

**No es un bloque del taller.** Se lee después, o nunca.

El notebook 06 cierra diciendo que, una vez que viste el bucle, un framework es
una conveniencia y no una caja negra. Aquí eso se mide en vez de afirmarse: el
agente del **paso 4** —el del inventario, con sus guardrails— vuelto a escribir
con LangChain, apuntando al mismo Ollama y con el mismo modelo.

## Por qué va al final y no al principio

Si esto apareciera en el bloque 2, el taller perdería su argumento. El valor de
`create_agent` es que **ya sabes qué hace por dentro**, porque escribiste ese
bucle a mano y lo viste romperse en el bloque 5. Sin eso, es magia que funciona
hasta que no.

Esa misma idea está escrita en tres lugares del repositorio, y conviene saber
que no es un adorno: en
[`agentlib/__init__.py`](../src/agentlib/__init__.py), en
[`agentlib/ollama.py`](../src/agentlib/ollama.py) y en el cierre del notebook
06.

## Qué necesita

```
.venvw\Scripts\python -m pip install -r requirements-frameworks.txt
```

**Son 81 paquetes**, contra los 3 del `requirements.txt` principal. Por eso
viven en un archivo aparte y por eso no se instala durante el taller: la red del
aula es un riesgo conocido.

Necesita además Ollama y el backend arriba, igual que los bloques 3 y 4.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

# Primero lo que puede faltar, con el mensaje que de verdad sirve. Un
# ImportError pelado de 'langchain' no le dice a nadie que hay un archivo de
# requirements aparte.
try:
    import langchain
    import langchain_ollama
    import langgraph
except ImportError as e:
    raise SystemExit(
        f"Falta {e.name}. Este notebook es el unico que necesita LangChain:\n"
        f"    python -m pip install -r ../requirements-frameworks.txt\n"
        f"Los cinco bloques del taller NO lo necesitan.")

import importlib.metadata as _md
for _p in ("langchain", "langchain-ollama", "langchain-core", "langgraph"):
    print(f"  {_p:18} {_md.version(_p)}")

  langchain          1.5.0
  langchain-ollama   1.1.0
  langchain-core     1.8.1
  langgraph          1.2.22


In [2]:
import importlib

import agentlib
import agentlib.ollama
for _m in (agentlib.ollama, agentlib):
    importlib.reload(_m)

from agentlib import cargar_env, base_url, modelo_configurado

_env, _claves, _viejas = cargar_env()
print(f"  .env    : {_env}  ({_claves} claves)")
print(f"  ollama  : {base_url()}")
print(f"  modelo  : {modelo_configurado()}")

# El agente escrito a mano, para comparar contra el.
import paso3_tools_http.agente as p3
import paso4_guardrails.agente as p4
importlib.reload(p3); importlib.reload(p4)

import os
import requests

_backend = os.getenv("BACKEND_URL", "http://localhost:8000")
try:
    print(f"  backend : {_backend}  ({requests.get(f'{_backend}/api/health', timeout=5).json()})")
except Exception as _e:
    print(f"  backend : NO RESPONDE ({type(_e).__name__}). Levantalo con docker compose up -d")

  .env    : <HOME>\Documents\desarrollo\tec\agenticai\.env  (19 claves)
  ollama  : http://10.49.x.x:11434
  modelo  : qwen3.5:9b


  backend : http://localhost:8000  ({'estado': 'ok', 'trazas': 0})


## La dirección del servidor no se escribe a mano

`ChatOllama` recibe un `base_url`, y aquí le llega el que resuelve
`agentlib.base_url()` desde `OLLAMA_DESTINO` del `.env`.

**Eso no es pulcritud.** Una celda con la IP del servidor escrita a mano
funciona para quien apunta a ese servidor y falla para todos los demás, y ese
error ya pasó antes en este repositorio.

In [3]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

# temperature=0 y reasoning=False: las mismas dos perillas que agentlib.chat()
# fija, para que la comparacion sea contra el mismo modelo y no contra otra
# configuracion.
llm = ChatOllama(base_url=base_url(), model=modelo_configurado(),
                 temperature=0, reasoning=False)

# Las tools son las MISMAS funciones del paso 3. El decorador @tool las envuelve
# y saca el esquema del nombre, los tipos y el docstring: eso ultimo es lo que
# en el paso 3 se escribe a mano como JSON.
EJECUCIONES = {}


def _contar(nombre):
    EJECUCIONES[nombre] = EJECUCIONES.get(nombre, 0) + 1


@tool
def consultar_catalogo() -> dict:
    """Catalogo de productos: clave, nombre y precio de cada uno."""
    _contar("consultar_catalogo")
    return p3.consultar_catalogo()


@tool
def consultar_stock(sku: str) -> dict:
    """Existencias y precio de un producto, por su clave."""
    _contar("consultar_stock")
    return p3.consultar_stock(sku)


TOOLS = [consultar_catalogo, consultar_stock]
print(f"  {len(TOOLS)} tools | esquema que LangChain saco del docstring:")
print(f"    {consultar_stock.name}: {consultar_stock.description}")
print(f"    args: {consultar_stock.args}")

  2 tools | esquema que LangChain saco del docstring:
    consultar_stock: Existencias y precio de un producto, por su clave.
    args: {'sku': {'title': 'Sku', 'type': 'string'}}


# 1. El mismo bucle, dos veces

La misma petición que el notebook 06 usa para la demo 1: el costo de todo el
inventario. Primero el agente escrito a mano.

In [4]:
# El del paso 4. rechazar_todo porque esta peticion no debe tocar nada.
from agentlib import llamadas_al_modelo

llamadas_al_modelo(reiniciar=True)
p4.correr("Cuanto cuesta llevarme TODO el inventario?", confirmar=p4.rechazar_todo)
A_MANO = llamadas_al_modelo(reiniciar=True)
print(f"\n  llamadas al modelo: {A_MANO}")


  --- vuelta 1 ---


  pide: consultar_catalogo({})


  resultado: {'productos': [{'sku': 'CAFE-500', 'nombre': 'Café molido 500g', 'precio': 180.0, 'categoria': 'cafe'}, {'sku': 'CAFE-250', 'nombre': 'Café molido 250g', 'precio': 95.0, 'categoria': 'cafe'}, {'sku': 'CAFE-GRANO-1K', 'nombre': 'Café en grano 1kg', 'precio': 340.0, 'categoria': 'cafe'}, {'sku': 'TE-200', 'nombre': 'Té verde 200g', 'precio': 95.0, 'categoria': 'te'}, {'sku': 'TE-MANZ-100', 'nombre': 'Té de manzanilla 100g', 'precio': 62.0, 'categoria': 'te'}, {'sku': 'AZUCAR-1K', 'nombre': 'Azúcar 1kg', 'precio': 38.0, 'categoria': 'endulzante'}, {'sku': 'MIEL-500', 'nombre': 'Miel de abeja 500g', 'precio': 120.0, 'categoria': 'endulzante'}, {'sku': 'LECHE-1L', 'nombre': 'Leche entera 1L', 'precio': 29.0, 'categoria': 'lacteo'}, {'sku': 'LECHE-DESL-1L', 'nombre': 'Leche deslactosada 1L', 'precio': 34.0, 'categoria': 'lacteo'}, {'sku': 'VASO-360', 'nombre': 'Vaso desechable 360ml (50 pz)', 'precio': 85.0, 'categoria': 'desechable'}], 'total': 10}

  --- vuelta 2 ---


  pide: consultar_stock({'sku': 'CAFE-500'})


  resultado: {'sku': 'CAFE-500', 'nombre': 'Café molido 500g', 'existencias': 17, 'precio': 180.0, 'categoria': 'cafe'}
  pide: consultar_stock({'sku': 'CAFE-250'})


  resultado: {'sku': 'CAFE-250', 'nombre': 'Café molido 250g', 'existencias': 34, 'precio': 95.0, 'categoria': 'cafe'}
  pide: consultar_stock({'sku': 'CAFE-GRANO-1K'})


... (5 outputs left out)

  resultado: {'sku': 'LECHE-1L', 'nombre': 'Leche entera 1L', 'existencias': 3, 'precio': 29.0, 'categoria': 'lacteo'}
  pide: consultar_stock({'sku': 'LECHE-DESL-1L'})


  resultado: {'sku': 'LECHE-DESL-1L', 'nombre': 'Leche deslactosada 1L', 'existencias': 19, 'precio': 34.0, 'categoria': 'lacteo'}
  pide: consultar_stock({'sku': 'VASO-360'})


  resultado: {'sku': 'VASO-360', 'nombre': 'Vaso desechable 360ml (50 pz)', 'existencias': 7, 'precio': 85.0, 'categoria': 'desechable'}

  --- vuelta 3 ---


  responde: Aquí está el cálculo del costo total para llevarse TODO el inventario:

- Café molido 500g: 17 unidades × $180 = $3,060
- Café molido 250g: 34 unidades × $95 = $3,230
- Café en grano 1kg: 8 unidades ×

  llamadas al modelo: 3


Ahora el mismo trabajo con `create_agent`. Son **tres líneas**: el modelo, las
tools y el `invoke`.

In [5]:
# El mismo trabajo, con el framework. Las llamadas al modelo NO las cuenta
# agentlib -LangChain habla con Ollama por su cuenta-, asi que se cuentan los
# AIMessage del historial, que es una llamada cada uno.
EJECUCIONES.clear()
agente = create_agent(llm, TOOLS)
r = agente.invoke({"messages": [("user", "Cuanto cuesta llevarme TODO el inventario?")]})

CON_FRAMEWORK = sum(1 for m in r["messages"] if type(m).__name__ == "AIMessage")
print(f"  mensajes en el historial : {len(r['messages'])}")
print(f"  llamadas al modelo       : {CON_FRAMEWORK}")
print(f"  tools ejecutadas         : {sum(EJECUCIONES.values())}  {EJECUCIONES}")
print(f"\n  a mano: {A_MANO} llamadas | con framework: {CON_FRAMEWORK} llamadas")
print(f"\n  respuesta: {r['messages'][-1].content[:150]}")

  mensajes en el historial : 15
  llamadas al modelo       : 3
  tools ejecutadas         : 11  {'consultar_catalogo': 1, 'consultar_stock': 10}

  a mano: 3 llamadas | con framework: 3 llamadas

  respuesta: Para calcular el costo total de llevarse todo el inventario disponible, he sumado el valor de cada producto basado en su cantidad actual y precio unit


**El bucle tiene la misma forma.** Las mismas llamadas al modelo, el mismo
reparto de tools por vuelta, y el mismo resultado. Lo que cambió es quién
escribió el `while`.

Y eso es lo que el taller afirma desde el `requirements.txt`: *"lo que los
frameworks agregan es conveniencia, no capacidad"*. La primera mitad de la frase
queda demostrada arriba. La segunda mitad es más interesante.

---

# 2. Los guardrails, uno por uno

El paso 4 pone cinco redes alrededor del bucle, y escribirlas es el contenido
del bloque 3. La pregunta es cuáles de ellas trae LangChain y cuáles sigues
escribiendo tú.

**El bucle y sus redes son 74 líneas de código en
[`paso4_guardrails/agente.py`](../src/paso4_guardrails/agente.py)** — contadas
el 2026-11-17: `correr()` 46, `validar()` 17 y las tres de confirmación 11.

## 2.1 La validación de argumentos

A mano es la función `validar()`: 17 líneas que revisan el SKU y que las
existencias sean un entero entre 0 y 10 000.

En LangChain el esquema de la tool **es** la validación. Se declara con
Pydantic y el framework lo aplica antes de llamar a tu función.

In [6]:
# El tope de 10000 del paso 4, declarado en vez de programado.
from pydantic import BaseModel, Field


class ArgsExistencias(BaseModel):
    sku: str = Field(description="la clave del producto")
    existencias: int = Field(ge=0, le=10000,
                             description="unidades; el tope del negocio es 10000")


@tool(args_schema=ArgsExistencias)
def actualizar_existencias(sku: str, existencias: int) -> dict:
    """Fija las existencias de un producto."""
    _contar("actualizar_existencias")
    return p3.actualizar_existencias(sku, existencias)


EJECUCIONES.clear()
r = create_agent(llm, [actualizar_existencias]).invoke(
    {"messages": [("user", "Pon 999999 unidades de CAFE-500.")]})

print(f"  la funcion se ejecuto: {EJECUCIONES.get('actualizar_existencias', 0)} veces")
for m in r["messages"]:
    if type(m).__name__ == "ToolMessage":
        print(f"  lo que volvio al modelo: {m.content[:140]}")
print(f"  respuesta: {r['messages'][-1].content[:130]}")

  la funcion se ejecuto: 0 veces
  lo que volvio al modelo: Error invoking tool 'actualizar_existencias' with kwargs {'sku': 'CAFE-500', 'existencias': 999999} with error:
 existencias: Input should b
  respuesta: Lo siento, no puedo establecer las existencias en 999999 unidades porque el límite máximo permitido por el sistema es de 10000 uni


La función no se ejecutó, el error volvió al modelo con el motivo, y el modelo
se corrigió solo. Es el mismo comportamiento de la demo 3 del notebook 06, con
una diferencia: **aquí el tope de 10 000 está declarado en el esquema**, en vez
de programado en un `if`.

> **Interpretación, discutible.** Lo declarativo gana cuando la regla es una
> restricción de tipo o de rango. `validar()` sigue siendo mejor sitio para una
> regla que depende del negocio y no del dato — y el tope de 10 000, de hecho,
> es de negocio: sale de conocer el almacén, no de conocer el modelo.

## 2.2 El tope de vueltas, y el que el paso 4 no tiene

La demo 1 del notebook 06 llega a una conclusión incómoda: el tope de vueltas
acota las llamadas al modelo y **no** los efectos, porque el modelo puede pedir
diez tools en una sola vuelta. `paso4` tiene `MAX_VUELTAS` y no tiene ningún
tope sobre las tools.

LangChain trae dos middleware, uno para cada cosa. Y resulta que **solo uno de
los dos sirve para lo que importa**.

In [7]:
# Los dos topes del framework, sobre la peticion que pide diez tools de un tiron.
from langchain.agents.middleware import (ModelCallLimitMiddleware,
                                         ToolCallLimitMiddleware)

PET = "Cuanto cuesta llevarme TODO el inventario?"

for etiqueta, mw in (
        ("sin tope", []),
        ("tope 2 al modelo", [ModelCallLimitMiddleware(
            run_limit=2, exit_behavior="end")]),
        ("tope 1 al modelo", [ModelCallLimitMiddleware(
            run_limit=1, exit_behavior="end")]),
        ("tope 3 a tool, end", [ToolCallLimitMiddleware(
            thread_limit=3, exit_behavior="end")]),
        ("tope 3 a tool, continue", [ToolCallLimitMiddleware(
            thread_limit=3, exit_behavior="continue")])):
    EJECUCIONES.clear()
    r = create_agent(llm, TOOLS, middleware=mw).invoke({"messages": [("user", PET)]})
    _ai = sum(1 for m in r["messages"] if type(m).__name__ == "AIMessage")
    print(f"  {etiqueta:24} {_ai} llamadas al modelo | "
          f"tools EJECUTADAS: {sum(EJECUCIONES.values())}")

  sin tope                 3 llamadas al modelo | tools EJECUTADAS: 11


  tope 2 al modelo         3 llamadas al modelo | tools EJECUTADAS: 11


  tope 1 al modelo         2 llamadas al modelo | tools EJECUTADAS: 1


  tope 3 a tool, end       3 llamadas al modelo | tools EJECUTADAS: 1


  tope 3 a tool, continue  3 llamadas al modelo | tools EJECUTADAS: 3


Esa última columna es la que importa, y es la que cuesta medir bien: **cuenta
las veces que la función de Python corrió de verdad**, no los mensajes del
historial. El historial trae un mensaje por cada tool que el framework
*bloqueó*, y contarlos da la cifra equivocada.

Lo medido el 2026-11-17 contra el servidor del Tec:

| | llamadas al modelo | tools ejecutadas |
|---|---|---|
| sin tope | 3 | 11 |
| `ModelCallLimitMiddleware(run_limit=2)` | 3 | **11** |
| `ModelCallLimitMiddleware(run_limit=1)` | 2 | 1 |
| `ToolCallLimitMiddleware(3, "end")` | 3 | **1** |
| `ToolCallLimitMiddleware(3, "continue")` | 3 | 3 |

Mira la segunda fila. **El tope de llamadas al modelo, puesto en 2, dejó
ejecutar once tools.** La segunda llamada pidió diez de un tirón, las diez
corrieron, y el tope cerró después.

Es exactamente el agujero de la demo 1 del notebook 06, reproducido dentro del
framework y por la misma razón: **un tope sobre las llamadas al modelo no acota
los efectos, porque los efectos no están donde se cuenta.** El único que los
acota es el de tool, y trae dos políticas: `end` corta la vuelta entera y
`continue` deja pasar las primeras y bloquea el resto.

**El framework tapa el agujero que el paso 4 tiene abierto.** Y solo sabes que
ese agujero te importa porque lo viste en la demo 1 del notebook 06.

## 2.3 La confirmación humana

A mano es `confirmar_en_consola`, y el bucle le pregunta antes de ejecutar
cualquier tool que esté en `DESTRUCTIVAS`.

En LangChain es `HumanInTheLoopMiddleware`, y funciona distinto por dentro: en
vez de bloquear esperando un `input()`, **interrumpe el grafo y devuelve el
control**. Para eso necesita un checkpointer, porque la interrupción tiene que
guardarse en algún lado.

In [8]:
# La confirmacion humana, como interrupcion. Pide un checkpointer: sin el, no
# hay donde guardar el estado del grafo a medio camino.
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


@tool
def borrar_producto(sku: str) -> dict:
    """DESTRUCTIVA: borra un producto del catalogo."""
    _contar("borrar_producto")
    return p3._pedir("DELETE", f"/api/inventario/{sku}")


EJECUCIONES.clear()
agente_con_red = create_agent(
    llm, [borrar_producto],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"borrar_producto": True})],
    checkpointer=InMemorySaver())

r = agente_con_red.invoke(
    {"messages": [("user", "Borra el producto VASO-360 del catalogo.")]},
    config={"configurable": {"thread_id": "demo-09"}})

print(f"  borrar_producto se ejecuto: {EJECUCIONES.get('borrar_producto', 0)} veces")
print(f"  el grafo quedo interrumpido: {bool(r.get('__interrupt__'))}")
if r.get("__interrupt__"):
    _p = r["__interrupt__"][0].value["action_requests"][0]
    print(f"  esperando aprobacion de: {_p['name']}({_p['args']})")

  borrar_producto se ejecuto: 0 veces
  el grafo quedo interrumpido: True
  esperando aprobacion de: borrar_producto({'sku': 'VASO-360'})


La tool **no se ejecutó** y el grafo quedó detenido esperando una decisión.

**Esta diferencia es de arquitectura, no de comodidad.** `confirmar_en_consola`
bloquea el proceso hasta que alguien teclee; la interrupción de LangGraph
devuelve el control y guarda el estado, así que la aprobación puede llegar
minutos después, desde otro proceso o desde una página web. Para un sistema de
verdad —donde el que aprueba no es quien corrió el script— esa es la forma
correcta, y escribirla a mano es bastante más que siete líneas.

## 2.4 Lo que el framework no trae

El paso 4 tiene una red que LangChain no incluye: **la guarda de repetición**.
En el bucle a mano es la lista `vistas`, y es la que se ve disparar en la demo 1
del notebook 06, cuando el agente pide cuatro veces el mismo stock y el programa
se niega a partir de la segunda.

Buscado el 2026-11-17 en `langchain.agents.middleware`: no hay nada con
`repeat`, `dedup`, `loop` ni `cache` en el nombre. La escribes tú, como
middleware propio.

In [9]:
# Lo que trae el catalogo de middleware, para que lo juzgues tu.
import langchain.agents.middleware as M

_nombres = sorted(n for n in dir(M) if n.endswith("Middleware"))
for _n in _nombres:
    print(f"  {_n}")
print(f"\n  {len(_nombres)} middleware disponibles")
_rep = [n for n in _nombres
        if any(k in n.lower() for k in ("repeat", "dedup", "repetic", "cache"))]
print(f"  middleware de repeticion: {_rep or 'NINGUNO'}")

  AgentMiddleware
  AutoModeMiddleware
  ContextEditingMiddleware
  FilesystemFileSearchMiddleware
  HumanInTheLoopMiddleware
  LLMToolSelectorMiddleware
  ModelCallLimitMiddleware
  ModelFallbackMiddleware
  ModelRefusalMiddleware
  ModelRetryMiddleware
  ModelRouterMiddleware
  PIIMiddleware
  ProviderToolSearchMiddleware
  ShellToolMiddleware
  SummarizationMiddleware
  TodoListMiddleware
  ToolCallLimitMiddleware
  ToolErrorMiddleware
  ToolRetryMiddleware

  19 middleware disponibles
  middleware de repeticion: NINGUNO


---

# 3. El balance

Medido el **2026-11-17** contra `qwen3.5:9b` en el servidor del Tec, con
`langchain 1.5.0`.

| la red del paso 4 | escrita a mano | en LangChain |
|---|---|---|
| lista blanca de tools | `DISPONIBLES`, un dict | implícita: solo las que registras |
| validación de argumentos | `validar()`, 17 líneas | el esquema Pydantic de la tool |
| el error que vuelve al modelo | escrito en el bucle | lo hace solo |
| tope de vueltas | `MAX_VUELTAS` | `ModelCallLimitMiddleware`, con el mismo agujero |
| **tope de llamadas a tool** | **no existe** | `ToolCallLimitMiddleware` |
| confirmación humana | `confirmar_en_consola` | `HumanInTheLoopMiddleware`, como interrupción |
| **guarda de repetición** | `vistas`, en el bucle | **no hay** |

Cinco de siete vienen resueltas, una de ellas mejor que la versión a mano, y una
falta.

## Lo que esto cambia del taller, y lo que no

**No cambia el orden.** Escribir el bucle sigue siendo lo que te permite leer esa
tabla. `ToolCallLimitMiddleware` es una línea de configuración con tres
políticas, y elegir entre `end` y `continue` es la decisión que la demo 1 del
notebook 06 te enseñó a ver. Sin esa demo, las tres opciones se leen igual de
bien y eliges la del ejemplo de la documentación.

**Sí cambia lo que uno diría al final.** La frase del `requirements.txt` —*"los
frameworks agregan conveniencia, no capacidad"*— se sostiene para el bucle, y se
queda corta para los guardrails.

El tope de tools y la confirmación por interrupción son capacidades que el paso
4 no tiene, no atajos para algo que ya tenía.

> **Interpretación, discutible.** Para un taller de cuatro horas, escribir el
> bucle a mano sigue siendo la decisión correcta: 74 líneas que se leen en
> pantalla contra 81 paquetes que hay que instalar. Para un sistema que va a
> producción, el cálculo se invierte en cuanto necesitas que la aprobación
> humana llegue desde otro proceso.

## Lo que no se midió aquí

- **El costo en tokens.** `agentlib.ultima_medicion()` lee los contadores que
  Ollama devuelve, y LangChain no los expone por el mismo camino. Las dos
  versiones hacen las mismas llamadas, pero no está comprobado que manden los
  mismos tokens.
- **El multiagente.** Esto compara el agente único del paso 4. El reparto en
  tres agentes del bloque 4 tiene su equivalente en LangGraph, y es otro
  notebook.
- **Los otros frameworks.** Nada de lo de arriba dice algo sobre los Agents SDK
  ni sobre nadie más.